In [1]:
# Import required libraries
import os
from pathlib import Path
from dotenv import load_dotenv
from unstract.llmwhisperer import LLMWhispererClientV2

# Load environment variables
load_dotenv(override=True)

# Get API key
LLMWHISPERER_API_KEY = os.getenv("LLMWHISPERER_API_KEY")

if not LLMWHISPERER_API_KEY:
    raise ValueError("LLMWHISPERER_API_KEY not found in environment variables")

In [2]:
# Define the extraction function
def extract_text_with_llmwhisperer(pdf_path: str, api_key: str) -> str:
    """Extract text from PDF using LLMWhisperer (preserves table layout and handles handwriting).
    
    Args:
        pdf_path: Path to the PDF file.
        api_key: LLMWhisperer API key.
        
    Returns:
        Extracted text with preserved layout.
    """
    print(f"[1/3] Extracting text from PDF using LLMWhisperer...")
    print(f"       (Layout-preserving extraction for better handwriting and table recognition)")
    
    client = LLMWhispererClientV2(api_key=api_key)
    
    result = client.whisper(
        file_path=pdf_path,
        wait_for_completion=True,
        wait_timeout=300,
        output_mode="layout_preserving",
    )
    
    if result and "extraction" in result:
        full_text = result["extraction"].get("result_text", "")
        page_count = result.get("extraction", {}).get("page_count", "unknown")
        print(f"       Done ({page_count} pages, {len(full_text):,} characters)")
        return full_text
    
    raise ValueError("LLMWhisperer did not return expected result structure")

In [3]:
# Define the path to the PDF
pdf_path = "docs/accord-insurance-handwritten-scanned-copy.pdf"

# Check if file exists
if not Path(pdf_path).exists():
    raise FileNotFoundError(f"PDF file not found: {pdf_path}")
else:
    print(f"Found PDF: {pdf_path}")
    print(f"File size: {Path(pdf_path).stat().st_size / 1024:.2f} KB")

Found PDF: docs/accord-insurance-handwritten-scanned-copy.pdf
File size: 579.29 KB


In [4]:
# Extract text from the scanned handwritten document
extracted_text = extract_text_with_llmwhisperer(pdf_path, LLMWHISPERER_API_KEY)

print("\n" + "="*80)
print("EXTRACTED TEXT PREVIEW (first 1000 characters)")
print("="*80)
print(extracted_text[:1000])
print("\n... (truncated)")
print(f"\nTotal extracted text length: {len(extracted_text):,} characters")

2026-01-30 06:59:42,135 - unstract.llmwhisperer.client_v2 - DEBUG - logging_level set to DEBUG
2026-01-30 06:59:42,136 - unstract.llmwhisperer.client_v2 - DEBUG - base_url set to https://llmwhisperer-api.us-central.unstract.com/api/v2
2026-01-30 06:59:42,136 - unstract.llmwhisperer.client_v2 - DEBUG - whisper called
2026-01-30 06:59:42,136 - unstract.llmwhisperer.client_v2 - DEBUG - api_url: https://llmwhisperer-api.us-central.unstract.com/api/v2/whisper
2026-01-30 06:59:42,136 - unstract.llmwhisperer.client_v2 - DEBUG - params: {'mode': 'form', 'output_mode': 'layout_preserving', 'page_seperator': '<<<', 'pages_to_extract': '', 'median_filter_size': 0, 'gaussian_blur_radius': 0, 'line_splitter_tolerance': 0.4, 'horizontal_stretch_factor': 1.0, 'mark_vertical_lines': False, 'mark_horizontal_lines': False, 'line_spitter_strategy': 'left-priority', 'add_line_nos': False, 'include_line_confidence': False, 'lang': 'eng', 'tag': 'default', 'filename': '', 'webhook_metadata': '', 'use_webhoo

[1/3] Extracting text from PDF using LLMWhisperer...
       (Layout-preserving extraction for better handwriting and table recognition)


2026-01-30 06:59:43,437 - unstract.llmwhisperer.client_v2 - DEBUG - whisper_status called
2026-01-30 06:59:43,437 - unstract.llmwhisperer.client_v2 - DEBUG - url: https://llmwhisperer-api.us-central.unstract.com/api/v2/whisper-status
2026-01-30 06:59:43,948 - unstract.llmwhisperer.client_v2 - DEBUG - Whisper-hash:56a82fe6|fb8a9ed2f7edb60bf234df241a22fbc4 | STATUS: processing...
2026-01-30 06:59:48,955 - unstract.llmwhisperer.client_v2 - DEBUG - whisper_status called
2026-01-30 06:59:48,956 - unstract.llmwhisperer.client_v2 - DEBUG - url: https://llmwhisperer-api.us-central.unstract.com/api/v2/whisper-status
2026-01-30 06:59:49,479 - unstract.llmwhisperer.client_v2 - DEBUG - Whisper-hash:56a82fe6|fb8a9ed2f7edb60bf234df241a22fbc4 | STATUS: processing...
2026-01-30 06:59:54,481 - unstract.llmwhisperer.client_v2 - DEBUG - whisper_status called
2026-01-30 06:59:54,482 - unstract.llmwhisperer.client_v2 - DEBUG - url: https://llmwhisperer-api.us-central.unstract.com/api/v2/whisper-status
2026

       Done (unknown pages, 8,573 characters)

EXTRACTED TEXT PREVIEW (first 1000 characters)


                ®                COMMERCIAL INSURANCE APPLICATION                                                  DATE (MM/DD/YYYY) 

                                            APPLICANT INFORMATION SECTION                                        03/02/2024 
                                                                                                                         NAIC CODE 
 AGENCY                                                            CARRIER 
                                                                          Fincorp          Insurance                     52678 
           Fincorp           insurance                             COMPANY POLICY OR PROGRAM NAME                    PROGRAM CODE 

                                                                            COMINS                                     23 
                                                     

In [5]:
# Optional: Save the extracted text to a file for further analysis
output_file = "extracted_text_handwritten.txt"

with open(output_file, 'w', encoding='utf-8') as f:
    f.write(extracted_text)

print(f"Extracted text saved to: {output_file}")

Extracted text saved to: extracted_text_handwritten.txt


In [6]:
# Optional: Use LangChain + OpenAI to analyze the extracted text
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Initialize OpenAI model
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Create a prompt for analyzing the insurance document
analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert at analyzing insurance documents. 
    Extract key information from the document text provided and structure it clearly.
    Focus on: policy holder details, policy numbers, coverage amounts, dates, and any other important information."""),
    ("user", "Please analyze this insurance document text and extract all key information:\n\n{text}")
])

# Create chain
chain = analysis_prompt | llm

# Analyze the document
print("[2/3] Analyzing document with GPT-4o...")
analysis = chain.invoke({"text": extracted_text})

print("\n" + "="*80)
print("DOCUMENT ANALYSIS")
print("="*80)
print(analysis.content)

[2/3] Analyzing document with GPT-4o...

DOCUMENT ANALYSIS
Here is the extracted key information from the provided insurance document:

**Policy Holder Details:**
- **Name (First Named Insured):** George Simon
- **Mailing Address:** 53B, Beach Ville Avenue, Florida
- **Business Phone:** 302567
- **Website Address:** www.aivent.com
- **Business Type:** Corporation
- **Subchapter:** "S" Corporation

**Policy Information:**
- **Agency:** Fincorp Insurance
- **Carrier:** Fincorp Insurance
- **NAIC Code:** 52678
- **Company Policy or Program Name:** COMINS
- **Program Code:** 23
- **Policy Number:** 7532685
- **Contact Name:** Simon
- **Contact Phone:** 23986582
- **Underwriter:** John Adams
- **Quote Status:** Issued

**Coverage Details:**
- **Business Auto Premium:** $3,000
- **Commercial General Liability Premium:** $4,000
- **Motor Carrier Premium:** $5,000

**Policy Dates:**
- **Proposed Effective Date:** 03/02/2028
- **Proposed Expiration Date:** Not specified

**Payment Information:*

In [8]:
# Import libraries for pytesseract benchmark
import time
import pytesseract
from pdf2image import convert_from_path

print("Setting up pytesseract benchmark...")

Setting up pytesseract benchmark...


In [9]:
# Extract text using pytesseract for comparison
print("[BENCHMARK] Extracting text with pytesseract...")
start_time = time.time()

# Convert PDF to images
images = convert_from_path(pdf_path)
print(f"Converted PDF to {len(images)} image(s)")

# Extract text from each page
pytesseract_text = ""
for i, image in enumerate(images):
    print(f"Processing page {i+1}/{len(images)}...")
    page_text = pytesseract.image_to_string(image)
    pytesseract_text += page_text + "\n\n"

pytesseract_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"PYTESSERACT EXTRACTION COMPLETE")
print(f"{'='*80}")
print(f"Processing time: {pytesseract_time:.2f} seconds")
print(f"Total characters extracted: {len(pytesseract_text):,}")
print(f"\nFirst 500 characters:")
print(pytesseract_text[:500])

[BENCHMARK] Extracting text with pytesseract...
Converted PDF to 1 image(s)
Processing page 1/1...

PYTESSERACT EXTRACTION COMPLETE
Processing time: 3.99 seconds
Total characters extracted: 2,791

First 500 characters:
ACORD COMMERCIAL INSURANCE APPLICATION DATE (MIDDAY)
4 ne APPLICANT INFORMATION SECTION o2 o2/ 2024,
AGENCY | CARRIER ag 61S
( i EFincsyve LNs arance 2
- VIR COY PAS AT ONCe COMPANY POLICY OR PROGRAM NAME PROGRAM CODE
Comrns 23
POLICY NUMBER
7.532.689"
CONTACT Simove UNDERWRITER UNDERWRITER OFFICE
eNom 23ABES EZ Tohn adovys
(A, to ae Tay ISSUE POLICY ) RENEW
ae ai scot BOUND (Give Date and/or Attach Copy):
CODE: SUBCODE: CHANGE ~— — AM
AGENCY CUSTOMER ID: CANCEL H PM
LINES OF BUSINESS
INDICATE L


In [10]:
# Compare results: LLMWhisperer vs Pytesseract
print("="*80)
print("COMPARISON: LLMWHISPERER vs PYTESSERACT")
print("="*80)

# Note: You need to capture the LLMWhisperer time from cell 3
# For now, let's compare the text characteristics
print(f"\nLLMWhisperer:")
print(f"  - Characters extracted: {len(extracted_text):,}")
print(f"  - Lines: {len(extracted_text.splitlines())}")

print(f"\nPytesseract:")
print(f"  - Characters extracted: {len(pytesseract_text):,}")
print(f"  - Lines: {len(pytesseract_text.splitlines())}")
print(f"  - Processing time: {pytesseract_time:.2f} seconds")

# Calculate similarity (rough estimate)
llm_lines = set(extracted_text.lower().split())
pyt_lines = set(pytesseract_text.lower().split())
common_words = llm_lines.intersection(pyt_lines)
if len(llm_lines) > 0:
    similarity = len(common_words) / len(llm_lines) * 100
    print(f"\nWord overlap: {similarity:.1f}%")

print("\n" + "="*80)
print("KEY DIFFERENCES")
print("="*80)
print("LLMWhisperer:")
print("  ✓ Better layout preservation")
print("  ✓ More accurate handwriting recognition")
print("  ✓ Structured table data")
print("\nPytesseract:")
print("  ✓ Faster processing")
print("  ✓ Free and open-source")
print("  ✗ Less accurate on handwritten text")
print("  ✗ Poor layout preservation")

COMPARISON: LLMWHISPERER vs PYTESSERACT

LLMWhisperer:
  - Characters extracted: 8,573
  - Lines: 103

Pytesseract:
  - Characters extracted: 2,791
  - Lines: 60
  - Processing time: 3.99 seconds

Word overlap: 80.1%

KEY DIFFERENCES
LLMWhisperer:
  ✓ Better layout preservation
  ✓ More accurate handwriting recognition
  ✓ Structured table data

Pytesseract:
  ✓ Faster processing
  ✓ Free and open-source
  ✗ Less accurate on handwritten text
  ✗ Poor layout preservation


In [11]:
# Save pytesseract output for comparison
pytesseract_output_file = "extracted_text_handwritten_pytesseract.txt"

with open(pytesseract_output_file, 'w', encoding='utf-8') as f:
    f.write(pytesseract_text)

print(f"Pytesseract output saved to: {pytesseract_output_file}")

Pytesseract output saved to: extracted_text_handwritten_pytesseract.txt
